<a href="https://colab.research.google.com/github/Otza02/satellite-img-segmentation/blob/main/notebooks/train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Otza02/satellite-img-segmentation.git
%cd satellite-img-segmentation
!pip install -e .

Cloning into 'satellite-img-segmentation'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (45/45), done.
remote: Total 88 (delta 32), reused 76 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 826.90 KiB | 11.02 MiB/s, done.
Resolving deltas: 100% (32/32), done.
/content/satellite-img-segmentation
Obtaining file:///content/satellite-img-segmentation
  Preparing metadata (setup.py) ... done
  Running setup.py develop for satelliteSegmentation


## -- Reiniciar session --

In [ ]:
%cd /content/satellite-img-segmentation
!unzip data/processed.zip -d data/processed

In [4]:
from satelliteSegmentation.config import Config
from satelliteSegmentation.dataset import load_data, SatelliteData
from satelliteSegmentation.models.unet import UNet
from satelliteSegmentation.utils import train_model, run_one_epoch

import torch
from matplotlib import pyplot as plt
from torch.utils.data import DataLoader

In [12]:
data_train = load_data("val")
data_val = load_data("test")

100%|██████████| 648/648 [00:01<00:00, 438.55it/s]


Dataset cargado:
X shape = torch.Size([648, 3, 120, 120])
Y shape = torch.Size([648, 1, 120, 120])


100%|██████████| 648/648 [00:01<00:00, 392.39it/s]


Dataset cargado:
X shape = torch.Size([648, 3, 120, 120])
Y shape = torch.Size([648, 1, 120, 120])


In [10]:
conf = Config("cuda" if torch.cuda.is_available() else "cpu", epochs=8)
conf

Config(device='cpu', kernel_size=3, stride=1, in_channels=3, hidden_channels=(64, 128, 256, 512), bottleneck_channels=1024, num_classes=7, epochs=8, lr=0.0001, patience=5, min_delta=0.001)

In [11]:
model = UNet(conf)
f"{sum([p.numel() for p in model.parameters()]):,}"

'31,043,911'

In [13]:
import torch
from torch.optim import Adam
from torch.cuda.amp import GradScaler
import torch.nn as nn
from tqdm import tqdm
from time import time

# Re-define run_one_epoch and train_model to handle CPU autocast issue

def run_one_epoch_fixed(
    model: nn.Module,
    dataloader: torch.utils.data.DataLoader,
    criterion: nn.Module,
    optimizer: Adam = None,
    scaler: GradScaler = None,
    config=None,
):
    is_training = optimizer is not None
    if is_training:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    for batch in tqdm(dataloader, desc="Training" if is_training else "Validation"):
        images, masks = batch
        images = images.to(config.device)
        masks = masks.to(config.device)

        with torch.enable_grad() if is_training else torch.inference_mode():
            # Apply autocast with float16 only if on a CUDA device
            if config.device == "cuda":
                with torch.autocast(device_type="cuda", dtype=torch.float16):
                    logits = model(images)
                    loss = criterion(logits, masks)
            else: # CPU device, use default float32 precision
                logits = model(images)
                loss = criterion(logits, masks)

        if is_training:
            if config.device == "cuda" and scaler is not None: # Use scaler only if on CUDA with AMP
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else: # CPU path or no scaler
                loss.backward()
                optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * images.size(0)

    return total_loss / len(dataloader.dataset)

def train_model_fixed(model, train_loader, val_loader, config):
    model.to(config.device)

    optimizer = Adam(model.parameters(), lr=config.lr)
    criterion = nn.CrossEntropyLoss()

    # Initialize GradScaler only if CUDA is available and AMP is intended
    scaler = GradScaler() if config.device == "cuda" else None

    history = {"train_loss": [], "val_loss": []}

    best_val_loss = float("inf")
    patience_counter = 0

    print(f"Training on device: {config.device}")

    for epoch in range(config.epochs):
        start = time()
        train_loss = run_one_epoch_fixed( # Call the fixed version
            model=model,
            dataloader=train_loader,
            criterion=criterion,
            optimizer=optimizer,
            scaler=scaler,
            config=config,
        )

        val_loss = run_one_epoch_fixed( # Call the fixed version
            model=model,
            dataloader=val_loader,
            criterion=criterion,
            config=config,
        )

        end = time()
        epoch_time = end - start

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(
            f"Epoch {epoch+1}/{config.epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Time: {epoch_time:.2f}s"
        )

        if val_loss < best_val_loss - config.min_delta:
            best_val_loss = val_loss
            patience_counter = 0
            torch.save(model.state_dict(), "best_model.pth")
            print("Model saved!")
        else:
            patience_counter += 1
            print(f"Patience: {patience_counter}/{config.patience}")
            if patience_counter >= config.patience:
                print("Early stopping!")
                break

    model.load_state_dict(torch.load("best_model.pth"))
    return model, history

# Now call the fixed train_model
model, history = train_model_fixed(model, train_loader, val_loader, conf)

RuntimeError: Input type (int) and bias type (c10::Half) should be the same